In [ ]:
from google.colab import drive
import os
import json
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(PROJECT_PATH, "outputs")
TABLE_PATH = os.path.join(PROJECT_PATH, "reports", "tables")
FIGURE_PATH = os.path.join(PROJECT_PATH, "reports", "figures")

os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

prediction_df = pd.read_parquet(
    os.path.join(
        OUTPUT_PATH,
        "phase6_test_predictions.parquet"
    )
)

results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase6_classical_model_results.csv"
    )
)

rolling_linear_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase6_rolling_linear_results.csv"
    )
)

print("Phase 6 results restored.")
print("Test predictions:", prediction_df.shape)

In [ ]:
print(prediction_df.columns.tolist())

display(
    prediction_df.head()
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

actual = prediction_df["AQI_t_plus_1"]

prediction_columns = {
    "Persistence Baseline": "Persistence_Prediction",
    "Linear Regression": "LinearRegression_Prediction",
    "Decision Tree": "DecisionTree_Prediction",
    "Random Forest": "RandomForest_Prediction",
    "XGBoost": "XGBoost_Prediction"
}

independent_results = []

for model_name, column in prediction_columns.items():

    pred = prediction_df[column]

    independent_results.append(
        {
            "Model": model_name,
            "MAE": mean_absolute_error(actual, pred),
            "RMSE": np.sqrt(
                mean_squared_error(actual, pred)
            ),
            "R2": r2_score(actual, pred)
        }
    )

independent_results = pd.DataFrame(
    independent_results
)

display(
    independent_results.sort_values("RMSE")
)

In [ ]:
current_aqi_mae = mean_absolute_error(
    actual,
    prediction_df["AQI"]
)

current_aqi_rmse = np.sqrt(
    mean_squared_error(
        actual,
        prediction_df["AQI"]
    )
)

current_aqi_r2 = r2_score(
    actual,
    prediction_df["AQI"]
)

print("Current AQI → next-hour AQI")
print("--------------------------------")
print("MAE :", round(current_aqi_mae, 6))
print("RMSE:", round(current_aqi_rmse, 6))
print("R²  :", round(current_aqi_r2, 6))

In [ ]:
prediction_df["AQI_change_1h"] = (
    prediction_df["AQI_t_plus_1"]
    - prediction_df["AQI"]
)

print(
    prediction_df["AQI_change_1h"].describe()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    prediction_df["AQI_change_1h"],
    bins=100
)

plt.title("Distribution of One-Hour AQI Change")
plt.xlabel("AQI(t+1) - AQI(t)")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "one_hour_aqi_change_distribution.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
feature_columns = [
    column
    for column in prediction_df.columns
    if column not in [
        "AQI_t_plus_1",
        "Persistence_Prediction",
        "LinearRegression_Prediction",
        "DecisionTree_Prediction",
        "RandomForest_Prediction",
        "XGBoost_Prediction",
        "AQI_change_1h"
    ]
]

print(
    "Prediction file does not contain the engineered feature matrix, "
    "so the direct leakage scan will be performed on Phase 5 data."
)

In [ ]:
PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

print(
    "Feature dataset:",
    feature_df.shape
)

In [ ]:
target_corr = (
    feature_df
    .select_dtypes(include=np.number)
    .corr()["AQI_t_plus_1"]
    .drop("AQI_t_plus_1")
    .abs()
    .sort_values(ascending=False)
)

display(
    target_corr.head(25).to_frame(
        "AbsoluteCorrelationWithFutureAQI"
    )
)

In [ ]:
suspicious_keywords = [
    "future",
    "next",
    "t_plus",
    "lead",
    "target"
]

suspicious_columns = [
    column
    for column in feature_df.columns
    if any(
        keyword in column.lower()
        for keyword in suspicious_keywords
    )
]

print(
    "Potentially suspicious feature names:"
)

for column in suspicious_columns:
    print("-", column)

In [ ]:
sample = feature_df[
    [
        "StationId",
        "Datetime",
        "AQI",
        "AQI_lag_1h",
        "AQI_lag_2h",
        "AQI_lag_12h"
    ]
].dropna(
    subset=["AQI_lag_1h"]
).head(20)

display(sample)

In [ ]:
rolling_check_columns = [
    "AQI",
    "AQI_rolling_mean_3h",
    "AQI_rolling_mean_12h",
    "PM2.5_rolling_mean_3h",
    "PM2.5_rolling_mean_12h"
]

display(
    feature_df[
        rolling_check_columns
    ].head(20)
)

In [ ]:
best_prediction_column = (
    independent_results
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

best_column = prediction_columns[
    best_prediction_column
]

station_results = (
    prediction_df
    .groupby(
        ["StationId", "StationName"]
    )
    .apply(
        lambda group: pd.Series({
            "Records": len(group),
            "MAE": mean_absolute_error(
                group["AQI_t_plus_1"],
                group[best_column]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["AQI_t_plus_1"],
                    group[best_column]
                )
            ),
            "R2": r2_score(
                group["AQI_t_plus_1"],
                group[best_column]
            )
        })
    )
    .sort_values("RMSE")
)

display(station_results)

In [ ]:
season_lookup = (
    feature_df[
        ["StationId", "Datetime", "Season"]
    ]
    .drop_duplicates(
        ["StationId", "Datetime"]
    )
)

season_test = prediction_df.merge(
    season_lookup,
    on=["StationId", "Datetime"],
    how="left",
    validate="one_to_one"
)

season_results = (
    season_test
    .groupby("Season")
    .apply(
        lambda group: pd.Series({
            "Records": len(group),
            "MAE": mean_absolute_error(
                group["AQI_t_plus_1"],
                group[best_column]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    group["AQI_t_plus_1"],
                    group[best_column]
                )
            ),
            "R2": r2_score(
                group["AQI_t_plus_1"],
                group[best_column]
            )
        })
    )
)

display(season_results)

In [ ]:
event_lookup = (
    feature_df[
        [
            "StationId",
            "Datetime",
            "IsStubbleSeason",
            "IsDiwaliWindow"
        ]
    ]
    .drop_duplicates(
        ["StationId", "Datetime"]
    )
)

event_test = prediction_df.merge(
    event_lookup,
    on=["StationId", "Datetime"],
    how="left",
    validate="one_to_one"
)

event_results = []

for event_column in [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]:

    for event_value in [0, 1]:

        subset = event_test[
            event_test[event_column] == event_value
        ]

        if len(subset) == 0:
            continue

        event_results.append(
            {
                "Event": event_column,
                "Value": event_value,
                "Records": len(subset),
                "MAE": mean_absolute_error(
                    subset["AQI_t_plus_1"],
                    subset[best_column]
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        subset["AQI_t_plus_1"],
                        subset[best_column]
                    )
                ),
                "R2": r2_score(
                    subset["AQI_t_plus_1"],
                    subset[best_column]
                )
            }
        )

event_results = pd.DataFrame(
    event_results
)

display(event_results)

In [ ]:
extreme_threshold = feature_df[
    "AQI"
].quantile(0.90)

extreme_test = prediction_df[
    prediction_df["AQI"] >= extreme_threshold
]

extreme_mae = mean_absolute_error(
    extreme_test["AQI_t_plus_1"],
    extreme_test[best_column]
)

extreme_rmse = np.sqrt(
    mean_squared_error(
        extreme_test["AQI_t_plus_1"],
        extreme_test[best_column]
    )
)

extreme_r2 = r2_score(
    extreme_test["AQI_t_plus_1"],
    extreme_test[best_column]
)

print("Extreme-period threshold:", extreme_threshold)
print("Extreme-period records:", len(extreme_test))
print("MAE :", round(extreme_mae, 4))
print("RMSE:", round(extreme_rmse, 4))
print("R²  :", round(extreme_r2, 4))

In [ ]:
sample_plot = prediction_df[
    [
        "Datetime",
        "AQI_t_plus_1",
        best_column
    ]
].sort_values("Datetime")

if len(sample_plot) > 5000:
    sample_plot = sample_plot.tail(5000)

plt.figure(figsize=(16, 6))

plt.plot(
    sample_plot["Datetime"],
    sample_plot["AQI_t_plus_1"],
    label="Actual",
    linewidth=1
)

plt.plot(
    sample_plot["Datetime"],
    sample_plot[best_column],
    label="Predicted",
    linewidth=1
)

plt.title(
    f"Actual vs Predicted Next-Hour AQI — {best_prediction_column}"
)

plt.xlabel("Time")
plt.ylabel("AQI")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "best_model_actual_vs_predicted.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
prediction_df["BestModelError"] = (
    prediction_df["AQI_t_plus_1"]
    - prediction_df[best_column]
)

plt.figure(figsize=(12, 6))

plt.hist(
    prediction_df["BestModelError"],
    bins=100
)

plt.title(
    f"Prediction Error Distribution — {best_prediction_column}"
)

plt.xlabel("Actual - Predicted AQI")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "best_model_error_distribution.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
ablation_a_features = [
    "AQI",
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

ablation_a_numeric = ablation_a_features
ablation_a_categorical = ["StationId"]

print(
    "Experiment A features:",
    len(ablation_a_features) + 1
)

In [ ]:
# Recreate the Phase 6 chronological split

unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[split_position - 1]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("Training period:")
print(
    train_df["Datetime"].min(),
    "→",
    train_df["Datetime"].max()
)

print("\nTest period:")
print(
    test_df["Datetime"].min(),
    "→",
    test_df["Datetime"].max()
)

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

In [ ]:
# Target variable
target = "AQI_t_plus_1"

# Ablation Experiment A features
ablation_a_features = [
    "AQI",
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

ablation_a_numeric = ablation_a_features
ablation_a_categorical = ["StationId"]

print("Target:", target)
print("Numeric features:", ablation_a_numeric)
print("Categorical features:", ablation_a_categorical)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

ablation_a_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            ablation_a_numeric
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=True
                        )
                    )
                ]
            ),
            ablation_a_categorical
        )
    ]
)

ablation_a_model = Pipeline(
    steps=[
        ("preprocessor", ablation_a_preprocessor),
        ("model", LinearRegression())
    ]
)

ablation_a_model.fit(
    train_df[
        ablation_a_numeric + ablation_a_categorical
    ],
    train_df[target]
)

print("Experiment A trained successfully.")

In [ ]:
ablation_a_pred = ablation_a_model.predict(
    test_df[
        ablation_a_numeric + ablation_a_categorical
    ]
)

ablation_a_mae = mean_absolute_error(
    test_df[target],
    ablation_a_pred
)

ablation_a_rmse = np.sqrt(
    mean_squared_error(
        test_df[target],
        ablation_a_pred
    )
)

ablation_a_r2 = r2_score(
    test_df[target],
    ablation_a_pred
)

print("Experiment A")
print("------------")
print("MAE :", round(ablation_a_mae, 6))
print("RMSE:", round(ablation_a_rmse, 6))
print("R²  :", round(ablation_a_r2, 6))

In [ ]:
# Recreate temporal feature groups for Phase 7

lag_features = [
    col
    for col in feature_df.columns
    if "_lag_" in col
]

rolling_features = [
    col
    for col in feature_df.columns
    if "rolling_" in col
]

print("Lag features:", len(lag_features))
print("Rolling features:", len(rolling_features))

print("\nLag feature examples:")
print(lag_features[:10])

print("\nRolling feature examples:")
print(rolling_features[:10])

In [ ]:
# --------------------------------------------------
# Recreate all Phase 7 ablation variables
# --------------------------------------------------

target = "AQI_t_plus_1"

# Experiment A:
# Current AQI + calendar + event + station
ablation_a_numeric = [
    "AQI",
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

ablation_a_categorical = [
    "StationId"
]

# Identify lag features
lag_features = [
    col
    for col in feature_df.columns
    if "_lag_" in col
]

# Identify rolling features
rolling_features = [
    col
    for col in feature_df.columns
    if "rolling_" in col
]

# Experiment B:
# Experiment A + lag + rolling features
ablation_b_numeric = (
    ablation_a_numeric
    + lag_features
    + rolling_features
)

print("=" * 60)
print("ABLATION FEATURES RESTORED")
print("=" * 60)

print("Target:", target)

print(
    "\nExperiment A numeric features:",
    len(ablation_a_numeric)
)

print(
    "Lag features:",
    len(lag_features)
)

print(
    "Rolling features:",
    len(rolling_features)
)

print(
    "Experiment B numeric features:",
    len(ablation_b_numeric)
)

print(
    "\nExperiment B feature list:"
)

print(ablation_b_numeric)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

ablation_b_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            ablation_b_numeric
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=True
                        )
                    )
                ]
            ),
            ablation_a_categorical
        )
    ]
)

ablation_b_model = Pipeline(
    steps=[
        ("preprocessor", ablation_b_preprocessor),
        ("model", LinearRegression())
    ]
)

ablation_b_model.fit(
    train_df[
        ablation_b_numeric
        + ablation_a_categorical
    ],
    train_df[target]
)

print("Experiment B trained successfully.")

In [ ]:
ablation_b_pred = ablation_b_model.predict(
    test_df[
        ablation_b_numeric
        + ablation_a_categorical
    ]
)

ablation_b_mae = mean_absolute_error(
    test_df[target],
    ablation_b_pred
)

ablation_b_rmse = np.sqrt(
    mean_squared_error(
        test_df[target],
        ablation_b_pred
    )
)

ablation_b_r2 = r2_score(
    test_df[target],
    ablation_b_pred
)

print("Experiment B")
print("------------")
print("MAE :", round(ablation_b_mae, 6))
print("RMSE:", round(ablation_b_rmse, 6))
print("R²  :", round(ablation_b_r2, 6))

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# --------------------------------------------------
# Recreate baseline metrics from saved Phase 6 data
# --------------------------------------------------

actual = prediction_df["AQI_t_plus_1"]

baseline_pred = prediction_df["Persistence_Prediction"]

baseline_mae = mean_absolute_error(
    actual,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        actual,
        baseline_pred
    )
)

baseline_r2 = r2_score(
    actual,
    baseline_pred
)

# --------------------------------------------------
# Verify Experiment A results exist
# --------------------------------------------------

print("Experiment A metrics:")
print("MAE :", round(ablation_a_mae, 6))
print("RMSE:", round(ablation_a_rmse, 6))
print("R²  :", round(ablation_a_r2, 6))

print("\nExperiment B metrics:")
print("MAE :", round(ablation_b_mae, 6))
print("RMSE:", round(ablation_b_rmse, 6))
print("R²  :", round(ablation_b_r2, 6))

# --------------------------------------------------
# Recreate full-model metrics directly
# --------------------------------------------------

full_model_row = independent_results.loc[
    independent_results["Model"] != "Persistence Baseline"
].sort_values(
    "RMSE",
    ascending=True
).iloc[0]

full_model_name = full_model_row["Model"]

full_model_mae = full_model_row["MAE"]
full_model_rmse = full_model_row["RMSE"]
full_model_r2 = full_model_row["R2"]

# --------------------------------------------------
# Build final ablation table
# --------------------------------------------------

ablation_results = pd.DataFrame({
    "Experiment": [
        "Persistence Baseline",
        "A: Current AQI + Calendar + Events + Station",
        "B: A + Lag + Rolling",
        f"Best Classical Model: {full_model_name}"
    ],

    "MAE": [
        baseline_mae,
        ablation_a_mae,
        ablation_b_mae,
        full_model_mae
    ],

    "RMSE": [
        baseline_rmse,
        ablation_a_rmse,
        ablation_b_rmse,
        full_model_rmse
    ],

    "R2": [
        baseline_r2,
        ablation_a_r2,
        ablation_b_r2,
        full_model_r2
    ]
})

display(ablation_results)

In [ ]:
ablation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    ),
    index=False
)

print("Ablation results saved successfully.")

In [ ]:
full_model_row = independent_results[
    independent_results["Model"] == best_prediction_column
].iloc[0]

ablation_results = pd.DataFrame({
    "Experiment": [
        "Persistence",
        "A: Current AQI + Calendar + Events + Station",
        "B: A + Lag + Rolling",
        f"Full Model: {best_prediction_column}"
    ],
    "MAE": [
        baseline_mae,
        ablation_a_mae,
        ablation_b_mae,
        full_model_row["MAE"]
    ],
    "RMSE": [
        baseline_rmse,
        ablation_a_rmse,
        ablation_b_rmse,
        full_model_row["RMSE"]
    ],
    "R2": [
        baseline_r2,
        ablation_a_r2,
        ablation_b_r2,
        full_model_row["R2"]
    ]
})

display(ablation_results)

In [ ]:
independent_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_independent_model_results.csv"
    ),
    index=False
)

station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_station_results.csv"
    )
)

season_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_season_results.csv"
    )
)

event_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_event_results.csv"
    ),
    index=False
)

ablation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    ),
    index=False
)

print("Phase 7 evaluation tables saved.")

In [ ]:
independent_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_independent_model_results.csv"
    ),
    index=False
)

station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_station_results.csv"
    )
)

season_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_season_results.csv"
    )
)

event_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_event_results.csv"
    ),
    index=False
)

ablation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    ),
    index=False
)

print("Phase 7 evaluation tables saved.")

In [ ]:
phase7_summary = {
    "completed_phase": "Phase 7 - Model Evaluation and Ablation Study",
    "next_phase": "Phase 8 - Deep Learning Models",

    "best_model": best_prediction_column,

    "best_model_mae": float(
        full_model_row["MAE"]
    ),

    "best_model_rmse": float(
        full_model_row["RMSE"]
    ),

    "best_model_r2": float(
        full_model_row["R2"]
    ),

    "current_aqi_baseline_r2": float(
        current_aqi_r2
    ),

    "one_hour_aqi_change_mean": float(
        prediction_df["AQI_change_1h"].mean()
    ),

    "one_hour_aqi_change_std": float(
        prediction_df["AQI_change_1h"].std()
    ),

    "extreme_threshold": float(
        extreme_threshold
    )
}

phase7_summary_path = os.path.join(
    OUTPUT_PATH,
    "phase7_summary.json"
)

with open(
    phase7_summary_path,
    "w"
) as f:
    import json
    json.dump(
        phase7_summary,
        f,
        indent=4
    )

print("=" * 60)
print("PHASE 7 — EVALUATION COMPLETE")
print("=" * 60)

print("Best classical model:", best_prediction_column)
print("Best MAE:", round(full_model_row["MAE"], 6))
print("Best RMSE:", round(full_model_row["RMSE"], 6))
print("Best R²:", round(full_model_row["R2"], 6))

print("\nNext phase: Phase 8 - Deep Learning Models")
print("=" * 60)

In [ ]:
import os
import json
import pandas as pd

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save Phase 7 evaluation tables
# --------------------------------------------------

independent_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_independent_model_results.csv"
    ),
    index=False
)

station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_station_results.csv"
    )
)

season_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_season_results.csv"
    )
)

event_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_event_results.csv"
    ),
    index=False
)

ablation_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 2. Save Phase 7 summary
# --------------------------------------------------

phase7_summary = {
    "completed_phase": "Phase 7 - Model Evaluation and Ablation Study",
    "next_phase": "Phase 8 - Deep Learning Models",

    "best_classical_model": "Random Forest",

    "best_mae": 0.014194,
    "best_rmse": 0.731606,
    "best_r2": 0.999966,

    "evaluation_focus": [
        "overall performance",
        "station-wise performance",
        "season-wise performance",
        "event-wise performance",
        "extreme-AQI performance",
        "leakage audit",
        "feature ablation"
    ]
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase7_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase7_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 3. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 7 - Model Evaluation and Ablation Study",
    "next_phase": "Phase 8 - Deep Learning Models"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Verify saved files
# --------------------------------------------------

files_to_check = [
    os.path.join(
        TABLE_PATH,
        "phase7_independent_model_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase7_station_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase7_season_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase7_event_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase7_ablation_results.csv"
    ),
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 7 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 7")
print("Next: Phase 8 - Deep Learning Models")